In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%cudf.pandas.profile
### cell 20 ###

def add_gap_rows_2(essay):
    import numpy as np

    cols_to_keep = [
        'discourse_start',
        'discourse_end',
        'discourse_type',
        'gap_length',
        'gap_end_length'
    ]
    # pull only the relevant rows and columns (runs on GPU via cudf.pandas extension)
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop=True)

    # compute the end of the previous segment
    prev_ends = df_essay['discourse_end'].shift(1)
    # create a GPU‐side positional index to avoid Python loops
    idx = df_essay.index.to_series()
    mask_between = (df_essay['gap_length'] > 0) & (idx > 0)

    # collect original and any "between" gap rows
    pieces = [df_essay]
    if mask_between.any():
        df_between = pd.DataFrame({
            'discourse_start': prev_ends[mask_between] + 1,
            'discourse_end':   df_essay['discourse_start'][mask_between] - 1,
            'discourse_type':  'Nothing',
            'gap_length':      np.nan,
            'gap_end_length':  np.nan
        })
        pieces.append(df_between)

    # concatenate and sort everything by start position
    df_essay = pd.concat(pieces, ignore_index=True)
    df_essay = df_essay.sort_values('discourse_start').reset_index(drop=True)

    # handle a trailing gap at the end if present
    last_gap_end = df_essay['gap_end_length'].iloc[-1]
    if last_gap_end > 0:
        start = df_essay['discourse_end'].iloc[-1] + 1
        end = start + last_gap_end
        df_tail = pd.DataFrame({
            'discourse_start': [start],
            'discourse_end':   [end],
            'discourse_type':  ['Nothing'],
            'gap_length':      [np.nan],
            'gap_end_length':  [np.nan]
        })
        df_essay = pd.concat([df_essay, df_tail], ignore_index=True)

    return df_essay